# BATIKMATCH — Exploratory Data Analysis (EDA)

**Image Retrieval Project | Orang 1: Data & Evaluasi**

Notebook ini berisi proses pemeriksaan dataset awal BatikMatch: informasi dataset, distribusi kelas, format dan resolusi gambar, ukuran file, visualisasi sampel, serta deteksi awal duplikasi.

**Dataset:** `data/raw/DATASET/DATASET/`  
**Output:** `outputs/figures/` dan `outputs/reports/`

> Jalankan notebook dari folder utama repository `batikmacth` agar konfigurasi path dapat menemukan folder `data/` dengan benar. Jalankan sel secara berurutan dari atas ke bawah.

## 1. Import Library
Library digunakan untuk membaca file gambar, menghitung hash, mengolah metadata, dan membuat visualisasi.

In [ ]:
import hashlib
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

## 2. Konfigurasi Path
Notebook ini disimpan di root repository. Karena itu, direktori kerja saat menjalankan notebook diatur ke root project.

In [ ]:
ROOT_DIR = Path.cwd()

DATA_DIR = ROOT_DIR / "data" / "raw" / "DATASET" / "DATASET"

OUTPUT_DIR = ROOT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
REPORT_DIR = OUTPUT_DIR / "reports"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

print("Root project :", ROOT_DIR)
print("Dataset      :", DATA_DIR)
print("Output       :", OUTPUT_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Folder dataset tidak ditemukan: {DATA_DIR}\n"
        "Pastikan notebook dijalankan dari folder utama repository."
    )

## 3. Fungsi Hash Gambar
SHA256 digunakan untuk mengidentifikasi file gambar yang memiliki konten identik.

In [ ]:
def calculate_hash(image_path):
    sha256 = hashlib.sha256()

    with open(image_path, "rb") as file:
        for chunk in iter(lambda: file.read(4096), b""):
            sha256.update(chunk)

    return sha256.hexdigest()

## 4. Scanning Dataset
Setiap gambar pada folder TRAIN dan TEST diperiksa. Metadata yang dicatat meliputi label, nama file, lokasi file, resolusi, format, mode warna, ukuran file, dan SHA256. Gambar yang gagal dibaca dicatat sebagai gambar rusak.

In [ ]:
def scan_dataset():
    records = []
    corrupted_images = []

    for split in ["TRAIN", "TEST"]:
        split_dir = DATA_DIR / split

        if not split_dir.exists():
            raise FileNotFoundError(f"Folder tidak ditemukan: {split_dir}")

        classes = sorted(
            folder for folder in split_dir.iterdir()
            if folder.is_dir()
        )

        for class_dir in classes:
            label = class_dir.name

            image_files = sorted(
                file for file in class_dir.rglob("*")
                if file.is_file()
                and file.suffix.lower() in IMAGE_EXTENSIONS
            )

            for image_path in image_files:
                try:
                    image_hash = calculate_hash(image_path)

                    with Image.open(image_path) as img:
                        img.verify()

                    with Image.open(image_path) as img:
                        width, height = img.size
                        image_format = img.format
                        image_mode = img.mode

                    file_size = image_path.stat().st_size

                    records.append({
                        "split": split,
                        "label": label,
                        "filename": image_path.name,
                        "path": str(image_path.relative_to(ROOT_DIR)),
                        "width": width,
                        "height": height,
                        "format": image_format,
                        "mode": image_mode,
                        "size_kb": round(file_size / 1024, 2),
                        "sha256": image_hash
                    })

                except Exception as error:
                    corrupted_images.append({
                        "split": split,
                        "label": label,
                        "filename": image_path.name,
                        "path": str(image_path),
                        "error": str(error)
                    })

    return pd.DataFrame(records), corrupted_images


print("=" * 60)
print("BATIKMATCH - EXPLORATORY DATA ANALYSIS")
print("=" * 60)

df, corrupted = scan_dataset()

if df.empty:
    raise ValueError(
        "Tidak ada gambar valid yang ditemukan. "
        "Periksa kembali struktur folder dataset."
    )

print("\n[1] INFORMASI DATASET")
print(f"Total gambar valid : {len(df)}")
print(f"Total kelas        : {df['label'].nunique()}")
print(f"Jumlah TRAIN       : {(df['split'] == 'TRAIN').sum()}")
print(f"Jumlah TEST        : {(df['split'] == 'TEST').sum()}")
print(f"Gambar rusak       : {len(corrupted)}")

## 5. Distribusi Kelas
Tabel berikut menunjukkan jumlah gambar pada setiap kelas untuk TRAIN dan TEST.

In [ ]:
distribution = pd.crosstab(df["label"], df["split"])
distribution = distribution.reindex(
    columns=["TRAIN", "TEST"],
    fill_value=0
)

print("[2] DISTRIBUSI KELAS")
display(distribution)

distribution.to_csv(REPORT_DIR / "distribusi_kelas.csv")

## 6. Format dan Mode Warna Gambar
Pemeriksaan ini memastikan format file dan mode warna yang digunakan dataset.

In [ ]:
print("[3] FORMAT GAMBAR")
display(df["format"].value_counts().to_frame("jumlah"))

print("[4] MODE WARNA")
display(df["mode"].value_counts().to_frame("jumlah"))

## 7. Analisis Resolusi Gambar
Menghitung banyaknya gambar untuk setiap kombinasi lebar dan tinggi.

In [ ]:
resolution = (
    df.groupby(["width", "height"])
    .size()
    .reset_index(name="jumlah")
    .sort_values("jumlah", ascending=False)
)

print("[5] RESOLUSI GAMBAR")
display(resolution)

resolution.to_csv(REPORT_DIR / "resolusi_gambar.csv", index=False)

## 8. Analisis Ukuran File
Statistik ukuran file dihitung dalam KB dan total ukuran dataset ditampilkan dalam MB.

In [ ]:
print("[6] STATISTIK UKURAN FILE")
display(df["size_kb"].describe().to_frame("ukuran_kb"))

print(
    f"Total ukuran gambar: "
    f"{df['size_kb'].sum() / 1024:.2f} MB"
)

## 9. Visualisasi Distribusi Kelas

In [ ]:
distribution_plot = distribution.sort_index()

ax = distribution_plot.plot(
    kind="bar",
    figsize=(13, 6),
    width=0.8
)

plt.title("Distribusi Dataset Batik Berdasarkan Kelas")
plt.xlabel("Kategori Motif Batik")
plt.ylabel("Jumlah Gambar")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Dataset")
plt.tight_layout()

plt.savefig(FIGURE_DIR / "distribusi_kelas.png", dpi=300)
plt.show()
plt.close()

## 10. Visualisasi Resolusi Gambar

In [ ]:
plt.figure(figsize=(9, 5))

plt.scatter(df["width"], df["height"], alpha=0.6)

plt.title("Distribusi Resolusi Gambar Batik")
plt.xlabel("Lebar Gambar (Pixel)")
plt.ylabel("Tinggi Gambar (Pixel)")
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(FIGURE_DIR / "resolusi_gambar.png", dpi=300)
plt.show()
plt.close()

## 11. Visualisasi Ukuran File

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(df["size_kb"], bins=20, edgecolor="black")

plt.title("Distribusi Ukuran File Gambar")
plt.xlabel("Ukuran File (KB)")
plt.ylabel("Frekuensi")
plt.tight_layout()

plt.savefig(FIGURE_DIR / "ukuran_file.png", dpi=300)
plt.show()
plt.close()

## 12. Visualisasi Sampel Motif Batik
Menampilkan satu contoh gambar TRAIN dari setiap kelas. Karena dataset memiliki 15 kelas, visualisasi disusun dalam grid 3 × 5.

In [ ]:
classes = sorted(df["label"].unique())

fig, axes = plt.subplots(3, 5, figsize=(16, 11))
axes = axes.flatten()

for i, label in enumerate(classes):
    sample = df[
        (df["label"] == label) &
        (df["split"] == "TRAIN")
    ].iloc[0]

    image_path = ROOT_DIR / sample["path"]

    with Image.open(image_path) as img:
        axes[i].imshow(img.convert("RGB"))

    axes[i].set_title(label, fontsize=9)
    axes[i].axis("off")

for i in range(len(classes), len(axes)):
    axes[i].axis("off")

plt.suptitle(
    "Contoh Motif Batik dari Setiap Kategori",
    fontsize=15
)
plt.tight_layout()

plt.savefig(FIGURE_DIR / "sampel_motif_batik.png", dpi=300)
plt.show()
plt.close()

## 13. Deteksi Awal Duplikasi Gambar
Duplikasi diperiksa dengan mengelompokkan gambar berdasarkan SHA256. Pemeriksaan ini merupakan deteksi awal; rincian pemeriksaan dan penanganan duplikasi dilanjutkan pada notebook/script khusus duplicate check dan data cleaning.

In [ ]:
duplicates = df[
    df.duplicated(subset="sha256", keep=False)
].copy()

duplicates = duplicates.sort_values("sha256")

duplicates.to_csv(
    REPORT_DIR / "duplikasi_gambar.csv",
    index=False
)

duplicate_group_count = duplicates["sha256"].nunique()

cross_split_groups = 0
for _, group in duplicates.groupby("sha256"):
    if group["split"].nunique() > 1:
        cross_split_groups += 1

print("[7] HASIL DETEKSI DUPLIKASI")
print(
    f"Jumlah file dalam kelompok duplikat: {len(duplicates)}"
)
print(f"Jumlah kelompok duplikat: {duplicate_group_count}")
print(
    "Kelompok duplikat lintas TRAIN-TEST: "
    f"{cross_split_groups}"
)

## 14. Menyimpan Metadata Dataset
Metadata seluruh gambar valid dan daftar gambar rusak disimpan dalam format CSV.

In [ ]:
df.to_csv(REPORT_DIR / "metadata_dataset.csv", index=False)

pd.DataFrame(corrupted).to_csv(
    REPORT_DIR / "gambar_rusak.csv",
    index=False
)

print("Metadata tersimpan:", REPORT_DIR / "metadata_dataset.csv")
print("Data gambar rusak tersimpan:", REPORT_DIR / "gambar_rusak.csv")

## 15. Ringkasan Hasil EDA
Berdasarkan hasil eksekusi EDA pada dataset awal, diperoleh ringkasan berikut. Nilai ini merupakan hasil run yang telah dilaporkan pada proses pengerjaan proyek.

In [ ]:
summary = pd.DataFrame({
    "Informasi": [
        "Total gambar valid",
        "Jumlah kelas",
        "Jumlah TRAIN",
        "Jumlah TEST",
        "Gambar rusak",
        "Format JPEG",
        "Mode RGB",
        "Resolusi 224 x 224",
        "Total ukuran dataset (MB)"
    ],
    "Hasil": [
        len(df),
        df["label"].nunique(),
        (df["split"] == "TRAIN").sum(),
        (df["split"] == "TEST").sum(),
        len(corrupted),
        (df["format"].str.upper() == "JPEG").sum(),
        (df["mode"] == "RGB").sum(),
        ((df["width"] == 224) & (df["height"] == 224)).sum(),
        round(df["size_kb"].sum() / 1024, 2)
    ]
})

display(summary)

## 16. Kesimpulan

EDA dilakukan untuk memahami kondisi dataset sebelum tahap pembersihan dan eksperimen image retrieval. Berdasarkan hasil run yang telah diperoleh:

- Dataset awal terdiri dari **1.350 gambar** dan **15 kelas**.
- Terdapat **1.050 gambar TRAIN** dan **300 gambar TEST**.
- Seluruh gambar valid yang diperiksa berformat JPEG, mode RGB, dan beresolusi 224 × 224 piksel.
- Tidak ditemukan gambar rusak.
- Pemeriksaan awal menemukan kelompok gambar duplikat, termasuk duplikasi lintas TRAIN dan TEST, sehingga diperlukan proses duplicate check dan data cleaning lebih lanjut.

File laporan dan visualisasi disimpan di folder `outputs/`.

**Catatan:** Notebook ini hanya mencakup EDA. Deteksi duplikasi lanjutan, data cleaning, validasi dataset bersih, dan baseline Color Histogram masing-masing dibuat dalam notebook terpisah agar mudah diperiksa.